# Modeling a Vehicle

This guide covers the process of creating a vehicle model in FASTSim, starting
with a built-in example, filling in known specifications, and then calibrating the model
against measured data.

## Start from an Existing Vehicle

The easiest starting point is to load a built-in vehicle resource and modify it for your
target vehicle. FASTSim includes several reference vehicles across a range of powertrain types.

To see what built-in vehicles are available:

In [2]:
import fastsim

# List all available built-in vehicle resource files
print(fastsim.Vehicle.list_resources())

[PosixPath('2012_Ford_Fusion.yaml'), PosixPath('2016 Nissan Leaf 30 kWh thrml.yaml'), PosixPath('2016_TOYOTA_Prius_Two.yaml'), PosixPath('2020 Chevrolet Bolt EV thrml.yaml'), PosixPath('2021_Hyundai_Sonata_Hybrid_Blue_thrml.yaml'), PosixPath('2022 Tesla Model 3 RWD thrml.yaml'), PosixPath('2022_Renault_Zoe_ZE50_R135.yaml'), PosixPath('2026_Chrysler_Pacifica_Select.yaml'), PosixPath('2026_Chrysler_Pacifica_Select_thrml.yaml')]


To load a built-in vehicle and save it as a starting point for your own model:

```python
# Load the closest built-in vehicle (optimally with the same powertrain type) as your target vehicle
veh = fastsim.Vehicle.from_resource("2012_Ford_Fusion.yaml")

# Save to a new file to use as your starting template
veh.to_file("my_vehicle.yaml")

# You can reload from the file at any time
veh = fastsim.Vehicle.from_file("my_vehicle.yaml")
```

You can then open `my_vehicle.yaml` and edit it directly. The file is human-readable YAML
with a structure matching the component hierarchy described in [](vehicle.md).

### Choosing the Right Starting Vehicle

Pick the built-in vehicle closest to your target in terms of powertrain type and vehicle class.
Here are some examples for passenger cars of various powertrain types:

| Powertrain | Example Resource |
|------------|------------------|
| Conventional (gas/diesel) | `2012_Ford_Fusion.yaml` |
| Hybrid Electric (HEV) | `2016_TOYOTA_Prius_Two.yaml` |
| Battery Electric (BEV) | `2022_Renault_Zoe_ZE50_R135.yaml` |
| Plug-in Hybrid (PHEV) | `2026_Chrysler_Pacifica_Select.yaml` |

---

## Parameterizing Vehicle Specifications

The first step is to fill in all the physical specifications you can find from public
sources: manufacturer datasheets, vehicle certification data, EPA window sticker values,
published test reports, or direct measurement.

<!-- TODO: review all typical values flagged below before publishing -->

### Vehicle-Level Parameters

These apply to all powertrain types:

- **`name`** : A descriptive label for the vehicle (e.g. `"2024 Toyota RAV4 Hybrid"`).
- **`year`** : Model year.
- **`mass_kilograms`** : Total vehicle mass (curb weight + estimated occupant/cargo mass).
  This is typically the most influential parameter for energy consumption.
- **`pwr_aux_base_watts`** : Constant auxiliary electrical load (HVAC if not using an HVAC thermal model, headlights,
  infotainment, etc.).

### Chassis Parameters

These govern the road load forces that determine how much power the vehicle needs to
maintain speed:

- **`drag_coef`** : Aerodynamic drag coefficient $C_d$. Published by manufacturers or
  measured in a wind tunnel. <!-- TODO: typical passenger car values: 0.25–0.40 -->
- **`frontal_area_square_meters`** : Frontal area in m². Can be estimated from vehicle
  dimensions or looked up in published sources.
- **`wheel_rr_coef`** : Rolling resistance coefficient.
- **`wheel_radius_meters`** : Loaded tire radius. Can be computed from the tire code
  (e.g. `225/50R17`) or measured directly. If `tire_code` is provided and `wheel_radius_meters`
  is omitted, FASTSim will compute the radius automatically.
- **`tire_code`** : Optional tire size string (e.g. `"225/50R17"`). If set, FASTSim parses
  it to compute `wheel_radius_meters` automatically. Either this or `wheel_radius_meters`
  must be supplied.
- **`num_wheels`** : Almost always 4 for passenger vehicles.
- **`drive_type`** : One of `FWD`, `RWD`, or `AWD`. Affects traction calculations.
- **`wheel_base_meters`** : Distance between front and rear axles.
- **`cg_height_meters`** : Center of gravity height.
- **`drive_axle_weight_frac`** : Fraction of vehicle weight on the drive axle. Affects
  traction. <!-- TODO: typical values: 0.59–0.65 (FWD cars tend toward the high end) -->
- **`wheel_fric_coef`** : Tire-road friction coefficient. 0.7 is usually reasonable.
- **`wheel_inertia_kilogram_square_meters`** : Rotational inertia of all wheels combined.

### Fuel Converter / Engine [`Conv`, `HEV`, `PHEV`]

- **`pwr_out_max_watts`** : Peak engine power output. This is the rated power from the
  manufacturer spec sheet.
- **`pwr_out_max_init_watts`** : Initial ramp-limited power, representing the engine's
  capability before it reaches full power. Typically set to a
  fraction of peak power.
- **`pwr_ramp_lag_seconds`** : Time constant for power ramping.
- **`pwr_idle_fuel_watts`** : Fuel power consumed at idle. Set to match measured idle fuel
  consumption if available. If unknown, leave as `0.0` and set a non-zero auxiliary load.
- **`eff_interp_from_pwr_out`** : Engine efficiency as a function of fractional power
  output (0.0–1.0). See [](#calibration) below.

### Fuel Storage / Tank [`Conv`, `HEV`, `PHEV`]

- **`energy_capacity_joules`** : Usable fuel tank capacity in joules.
- **`pwr_out_max_watts`** : Maximum fuel power delivery rate. Can be left very large since it is rarely a binding constraint.
- **`pwr_ramp_lag_seconds`** : Power ramp lag for fuel delivery.

### Rechargeable Energy Storage / Battery [`BEV`, `HEV`, `PHEV`]

- **`energy_capacity_joules`** : Total usable battery energy capacity. Convert from kWh:
  $E = \text{kWh} \times 3.6 \times 10^6$.
- **`pwr_out_max_watts`** : Peak battery discharge power. Can be left very large if not
  the constraining factor.
- **`min_soc` / `max_soc`** : State-of-charge limits (0.0–1.0). Set based on manufacturer
  usable range (e.g. 0.1–0.9 for a 10–90% operational window).
- **`eff_interp`** : Battery efficiency. A constant value can be provided if measured data is unavailable.

### Electric Motor [`BEV`, `HEV`, `PHEV`]

- **`pwr_out_max_watts`** : Peak motor power output.
- **`eff_interp_achieved`** : Motor efficiency as a function of fractional power output.

### Transmission

- **`eff_interp`** : Transmission efficiency. A constant value is typically sufficient.

### Mass Handling

Vehicle mass can be specified at the top level (`mass_kilograms`) or split across
individual components. If all component-level `mass_kilograms` fields are `~` (null),
the top-level `mass_kilograms` is used as the total vehicle mass directly.

For component-level mass accounting, set `mass_kilograms` or
`specific_energy_joules_per_kilogram` / `specific_pwr_watts_per_kilogram` on
individual components. The vehicle-level mass then becomes the sum of all components plus
the glider mass (`chassis.glider_mass_kilograms`).

---

## Calibration

After populating all known specifications, the model should be validated against measured
fuel economy or energy consumption data before it is used for analysis.

### Comparing to Label Fuel Economy

One method of calibrating a vehicle is comparing simulation against the EPA window sticker values. For details on running this comparison, see [](../running-simulations/label-fe.ipynb).

### Fuel Converter / Electric Machine Efficiency Curves

The component efficiency curves have a large impact on simulated fuel economy.
The curve maps fractional power output (1.0 = 100% of max
power) to thermal efficiency (fraction of fuel energy converted to shaft power).

A default curve shape from a similar vehicle is a reasonable starting point. Calibrate from BSFC/efficiency maps in literature, operating points from real-world/chassis dynamometer data (such as [Argonne's D3 database](https://www.anl.gov/taps/downloadable-dynamometer-database)), etc.

### Road Load Validation

If coast-down test data or dynamometer road load target coefficients (ABCs) are available (e.g. from [EPA data](https://www.epa.gov/compliance-and-fuel-economy-data/data-cars-used-testing-fuel-economy)), use them to verify the chassis parameters (`drag_coef`,
`frontal_area_square_meters`, `wheel_rr_coef`) produce the correct road load at typical
highway speeds.

### Fine-Tuning Models

If you find the model under- or over-predicts differently on various cycles, ask if the cycles have distinct driving behavior. Are there higher speeds segments that indicate issues with the simulated road load? Are acceleration events matching measured data especially poorly? Questions like these can help identify parameters that need refinement. 